# Optimisation de profit d'un atelier en utilisant QAOA



## Exposition du probleme

Un atelier possede certaines **resources**:

- Machines
- Ouvriers
- Creneaux horraires

Un atelier doit prendre une **decision**:

- A quelles commandes repondre ? 

Un atelier a un objectif:

- Un atelier doit choisir ses commandes pour **maximiser ses profits**

Un atelier a une regle:

- Il est impossible de repondre a 2 commandes necesitant les meme resources.

## Translation mathematique du probleme

Ce probleme correspond au [Problème du stable maximum](https://fr.wikipedia.org/wiki/Probl%C3%A8me_du_stable_maximum) avec poids (**Maximum Weight Independent Set / MWIS**):

> En informatique théorique, le problème du stable maximum ou maximum independent set problem en anglais, est un problème d'optimisation qui consiste étant donné un graphe non orienté à trouver un stable de cardinal maximum, c'est-à-dire un sous-ensemble de sommets du graphe, le plus grand possible, tel que les éléments de ce sous-ensemble ne soient pas voisins.
>> https://fr.wikipedia.org/wiki/Probl%C3%A8me_du_stable_maximum

Tout les probleme MWIS sont **NP-Hard**:    
il y a $2^n$ candidats pour n commandes, puisque chaque commande peut avoir un des 2 etats booleens: acceptee ou refusee.    
Une commande sera acceptee si elle maximise le profit et ne cree pas dans de conflict.

## Definition des variables

Pour notre exemple, nous allons definir un atelier tel que:

- 4 machines
- 3 ouvriers
- 3 creneaux horraires

Qui doivent repondre a 8 commandes.

Voici les definitions du probleme en python:

In [23]:
# Tableau 1 :
# les ressources de l'atelier (nom -> type).
# Chacune a une capacité de 1 commande a la fois.
RESSOURCES: dict[str, str] = {
    "Fraiseuse CNC":      "machine",
    "Découpeuse laser":   "machine",
    "Cabine de peinture": "machine",
    "Tour":               "machine",
    "Soudeur certifié":   "ouvrier",
    "Inspecteur qualité": "ouvrier",
    "Électricien":        "ouvrier",
    "Créneau lundi":      "créneau",
    "Créneau mercredi":   "créneau",
    "Créneau vendredi":   "créneau",
}
print("Ressources:", RESSOURCES)

Ressources: {'Fraiseuse CNC': 'machine', 'Découpeuse laser': 'machine', 'Cabine de peinture': 'machine', 'Tour': 'machine', 'Soudeur certifié': 'ouvrier', 'Inspecteur qualité': 'ouvrier', 'Électricien': 'ouvrier', 'Créneau lundi': 'créneau', 'Créneau mercredi': 'créneau', 'Créneau vendredi': 'créneau'}


In [28]:
# Tableau 2 :
# La liste de profit des commandes.
# La commande n a pour profit COMMANDES[n - 1]
# Il y a autant de commande que d'element dans ce tableau.
COMMANDES: list[int] = [
    5, # Commande 1 a pour profit 5
    3,
    4,
    6,
    2,
    5,
    3,
    4
]
print(f"{len(COMMANDES)} commandes au profit:", COMMANDES)

8 commandes au profit: [5, 3, 4, 6, 2, 5, 3, 4]


In [25]:
# Tableau 3 : les conflits (ressource -> commandes qui en ont besoin).
# Deux commandes de la même ligne ne peuvent pas être acceptées ensemble.
CONFLITS: dict[str, list[int]] = {
    "Fraiseuse CNC":      [1, 2],
    "Découpeuse laser":   [3, 4],
    "Cabine de peinture": [5, 6],
    "Tour":               [7, 8],
    "Soudeur certifié":   [1, 3],
    "Inspecteur qualité": [4, 6],
    "Électricien":        [5, 7],
    "Créneau lundi":      [2, 4],
    "Créneau mercredi":   [3, 5],
    "Créneau vendredi":   [6, 8],
}

def conflits_pretty_print() -> None:
    largeur_nom: int = max(map(len, CONFLITS))
    largeur_type: int = max(map(len, RESSOURCES.values()))
    for ressource, commandes in CONFLITS.items():
        liste: str = " et ".join(f"n°{c} (profit {COMMANDES[c - 1]})" for c in commandes)
        print(f"{ressource:<{largeur_nom}}  [{RESSOURCES[ressource]:<{largeur_type}}]  ->  {liste}")

conflits_pretty_print()

Fraiseuse CNC       [machine]  ->  n°1 (profit 5) et n°2 (profit 3)
Découpeuse laser    [machine]  ->  n°3 (profit 4) et n°4 (profit 6)
Cabine de peinture  [machine]  ->  n°5 (profit 2) et n°6 (profit 5)
Tour                [machine]  ->  n°7 (profit 3) et n°8 (profit 4)
Soudeur certifié    [ouvrier]  ->  n°1 (profit 5) et n°3 (profit 4)
Inspecteur qualité  [ouvrier]  ->  n°4 (profit 6) et n°6 (profit 5)
Électricien         [ouvrier]  ->  n°5 (profit 2) et n°7 (profit 3)
Créneau lundi       [créneau]  ->  n°2 (profit 3) et n°4 (profit 6)
Créneau mercredi    [créneau]  ->  n°3 (profit 4) et n°5 (profit 2)
Créneau vendredi    [créneau]  ->  n°6 (profit 5) et n°8 (profit 4)


Ici, les verifications des donnees et la creation des `PAIRES` de conflits:

In [ ]:
# Assertion des listes et des creation de la liste de conflits:
from itertools import combinations

# Vérifie que les trois tableaux parlent des mêmes ressources et commandes.
def verifier_coherence(
    ressources: dict[str, str],
    commandes: list[int],
    conflits: dict[str, list[int]],
) -> None:
    assert set(conflits) == set(ressources), "ressource inconnue ou oubliée dans CONFLITS"
    n: int = len(commandes)
    for ressource, numeros in conflits.items():
        for c in numeros:
            assert 1 <= c <= n, f"commande inconnue : {c} (ressource {ressource})"

# Retourne les paires de commandes qui ne peuvent pas être acceptées ensemble.
def calculer_paires(conflits: dict[str, list[int]]) -> list[tuple[int, int]]:
    paires: list[tuple[int, int]] = []
    for numeros in conflits.values():
        for paire in combinations(sorted(numeros), 2):
            if paire not in paires:  # évite un doublon si deux commandes partagent 2 ressources
                paires.append(paire)
    return sorted(paires)

# Retourne les ressources dont la commande c a besoin
def ressources_de_commande(c: int, conflits: dict[str, list[int]]) -> list[str]:
    return [ressource for ressource, numeros in conflits.items() if c in numeros]

# Affiche pour chaque commande son profit et ses ressources.
def afficher_commandes(commandes: list[int], conflits: dict[str, list[int]]) -> None:
    for c in range(1, len(commandes) + 1):
        besoins: str = ", ".join(ressources_de_commande(c, conflits))
        print(f"Commande {c} (profit {commandes[c - 1]}) : {besoins}")

verifier_coherence(RESSOURCES, COMMANDES, CONFLITS)
N: int = len(COMMANDES)
PAIRES: list[tuple[int, int]] = calculer_paires(CONFLITS)

afficher_commandes(COMMANDES, CONFLITS)
print(f"\n{len(PAIRES)} paires en conflit : {PAIRES}")

Commande 1 (profit 5) : Fraiseuse CNC, Soudeur certifié
Commande 2 (profit 3) : Fraiseuse CNC, Créneau lundi
Commande 3 (profit 4) : Découpeuse laser, Soudeur certifié, Créneau mercredi
Commande 4 (profit 6) : Découpeuse laser, Inspecteur qualité, Créneau lundi
Commande 5 (profit 2) : Cabine de peinture, Électricien, Créneau mercredi
Commande 6 (profit 5) : Cabine de peinture, Inspecteur qualité, Créneau vendredi
Commande 7 (profit 3) : Tour, Électricien
Commande 8 (profit 4) : Tour, Créneau vendredi

10 paires en conflit : [(1, 2), (1, 3), (2, 4), (3, 4), (3, 5), (4, 6), (5, 6), (5, 7), (6, 8), (7, 8)]


## Resolution du probleme (Bruteforce)

En se basant sur les tableaux et valeurs d'exemples definis precedement, nous pouvons resoudre le probleme avec un algorithme exhaustif qui itere sur les 2^n possibilites:

In [72]:
from itertools import product
from typing import NamedTuple

class Resultat(NamedTuple):
    acceptees: list[int]  # numéros (à partir de 1) des commandes acceptées
    profit: int
    nb_testes: int        # choix examinés : 2^N
    nb_valides: int       # choix sans aucun conflit

# Un choix est une suite de 0 et de 1 : choix[i - 1] vaut 1 si la commande i est acceptée.
# Il est valide si aucune paire en conflit n'est acceptée en même temps.
def est_valide(choix: tuple[int, ...], paires: list[tuple[int, int]]) -> bool:
    return all(not (choix[i - 1] and choix[j - 1]) for i, j in paires)

# Somme des profits des commandes acceptées.
def calculer_profit(choix: tuple[int, ...], commandes: list[int]) -> int:
    return sum(profit for profit, accepte in zip(commandes, choix) if accepte)

# Teste les 2^N choix possibles et garde le valide qui rapporte le plus.
def recherche_exhaustive(commandes: list[int], paires: list[tuple[int, int]]) -> Resultat:
    meilleur_choix: tuple[int, ...] = (0,) * len(commandes)
    meilleur_profit: int = 0
    nb_testes: int = 0
    nb_valides: int = 0
    for choix in product((0, 1), repeat=len(commandes)):
        nb_testes += 1
        if not est_valide(choix, paires):
            continue
        nb_valides += 1
        profit: int = calculer_profit(choix, commandes)
        if profit > meilleur_profit:
            meilleur_choix, meilleur_profit = choix, profit
    acceptees: list[int] = [i + 1 for i, x in enumerate(meilleur_choix) if x]
    return Resultat(acceptees, meilleur_profit, nb_testes, nb_valides)

print("Defini:", est_valide, calculer_profit, recherche_exhaustive)

Defini: <function est_valide at 0x7d5b6c790180> <function calculer_profit at 0x7d5b6c790220> <function recherche_exhaustive at 0x7d5b6d119ee0>


In [95]:
# Execution des fonctions de recherches exhaustives
# Avec monitoring pour pouvoir mesurer les resultats.
from time import perf_counter

# Exécute la force brute `repetitions` fois et retourne le résultat avec la durée moyenne (en secondes).
def mesurer_recherche_exhaustive(
    commandes: list[int], paires: list[tuple[int, int]], repetitions: int
) -> tuple[Resultat, float]:
    assert repetitions >= 1, "repetitions doit être au moins 1"
    durees: list[float] = []
    for _ in range(repetitions):
        debut: float = perf_counter()
        resultat: Resultat = recherche_exhaustive(commandes, paires)
        durees.append(perf_counter() - debut)
    return resultat, sum(durees) / repetitions

repetitions=10000
resultat, duree_moyenne = mesurer_recherche_exhaustive(COMMANDES, PAIRES, repetitions=repetitions)
print(f"Meilleure sélection : commandes {resultat.acceptees}, profit {resultat.profit}")
print(f"Durée moyenne sur {repetitions} exécutions : {duree_moyenne * 1000:.3f} ms")

Meilleure sélection : commandes [1, 4, 5, 8], profit 17
Durée moyenne sur 10000 exécutions : 0.353 ms


### Resultat de la recherche exhaustive

Nous utiliserons ces resultats pour comparer avec QAOA.

```
Meilleure sélection : commandes [1, 4, 5, 8], profit 17
Durée moyenne sur 10000 exécutions : 0.353 ms
```